# Day 24 — Trees vs sequences on the same windows

Status: COMPLETE — LightGBM (96 feats, NaN-native, full set A) vs 1-layer LSTM
(24h × 26 channels incl. masks, all positives + 10x negatives). Compared on the
identical sampled test set. 12/12 tests pass; both scripts lint-clean.

In [1]:
import json
from pathlib import Path

lgbm = json.loads(Path("../models/lgbm_metrics.json").read_text())
cmp = json.loads(Path("../models/compare_metrics.json").read_text())
print(f"LightGBM full test : ROC-AUC {lgbm['roc_auc_at_0.5']} | "
      f"PR-AUC {lgbm['pr_auc_at_0.5']} | F1@0.5 {lgbm['f1_at_0.5']}")
print("  (baseline was 0.7237 / 0.0060 / 0.0093 — trees + NaN-native roughly double F1)")
print("On the SHARED sampled test set (~9% positive by construction):")
for name in ("lgbm_sampled", "lstm_sampled"):
    m = cmp[name]
    extra = f" | F1@0.5 {m['f1']}" if "f1" in m else ""
    print(f"  {name.replace('_sampled','').upper():<10}: "
          f"ROC-AUC {m['roc_auc']} | PR-AUC {m['pr_auc']}{extra}")

LightGBM full test : ROC-AUC 0.7329 | PR-AUC 0.0078 | F1@0.5 0.0172
  (baseline was 0.7237 / 0.0060 / 0.0093 — trees + NaN-native roughly double F1)
On the SHARED sampled test set (~9% positive by construction):
  LightGBM : ROC-AUC 0.7361 | PR-AUC 0.2439 | F1@0.5 0.3049
  LSTM     : ROC-AUC 0.7389 | PR-AUC 0.2139


## Verdict (read carefully — two comparisons, two stories)

1. **Ranking is a tie** (0.739 vs 0.736): the LSTM learned the temporal patterns
   from scratch about as well as hand-built rolling features rank them.
2. **LightGBM wins where it counts** (PR-AUC 0.244 vs 0.214 on the shared set;
   F1 doubled vs the LR baseline on full test). With limited positives and
   strong engineered features, trees generalize better — the sequence model
   would need more data, tuning, and architecture to pull ahead.
3. **Do NOT compare sampled PR-AUC (~0.2) with full-test PR-AUC (~0.008).**
   The sample is enriched to ~9% positive by construction; prevalence sets the
   PR baseline. Only same-rows comparisons are valid — which is why
   `compare_metrics.json` exists.

## Handoff to Day 25

Winner for deployment: **LightGBM**. Next: time-respecting CV
(forward-chaining — no future folds validating the past) plus threshold
tuning for the alert operating point, where the real gains hide.